# Baseline de la cátedra (diapositiva) con el código de la plantilla z495
Mismo código y mismo orden de celdas que la plantilla de la cátedra (Tarea para el Hogar 04: `mlr3mbo`, `particionar`, `loguear`). Todo lo demás sigue la diapositiva *Baseline*:

| Diapositiva | Cómo queda en esta notebook |
|---|---|
| 762 columnas para entrenar | Se imprime la cantidad real antes de la optimización |
| Data quality | En cada `foto_mes`, variable con todos los valores en cero y sin NA pasa a NA |
| FE histórico | lag1, lag2, delta1 y delta2 de todas las variables menos `numero_de_cliente`, `foto_mes` y `clase_ternaria` |
| Búsqueda | `semilla_primigenia` 585581, `ksemillerio` 1, `repe` 1, `iteraciones` 10, `undersampling` 0,1, `training_pct` 70 |
| Fijos | `colsample_bytree` 0,5, `colsample_bynode` 0,2; demás fijos desde el YAML |
| Bordes | `nrounds` 64–4096, `min_child_weight` 1e-6–0,1, `learning_rate` 0,001–0,05; `max_leaves` 16–2048 o `max_depth` 2–14, según lo que se declare en el YAML |
| Producción | Entrena con 202103+202104, predice 202106, **10 semillas**, `undersampling` 0,1 |
| A reportar | AUC, mejores hiperparámetros, ganancia máxima y cantidad de envíos |

**Cambios respecto de z495 (todos para respetar la diapositiva):**
- Sin Salsa Mágica: no se sacan `cprestamos_personales` ni `mprestamos_personales`.
- Se entrena con 202103 y 202104 (la plantilla usaba solo 202104) y se agregan data quality y lags/deltas.
- Se agrega `colsample_bynode` = 0,2 a los parámetros fijos.
- La validación es el 30 % limpio (`fold == 2`, ~97.800 filas) y no `training == 0`: este último incluye también los CONTINUA que el undersampling descartó del 70 % de entrenamiento (~300.900 filas).
- La métrica de la búsqueda es `auc` (la plantilla usaba `average_precision`).
- La plantilla pedía `n_evals = 10`, pero el diseño inicial de `mlr3mbo` ya evalúa 12 configuraciones y la búsqueda terminaba ahí, sin ninguna iteración bayesiana. Acá el terminador es 12 + 10 = 22 evaluaciones y se imprime cuántas se hicieron.
- Producción con 10 semillas (se promedian las probabilidades). Las semillas son 10 primos sorteados a partir de la semilla primigenia.
- Corre en local (sin Colab ni Drive).
- Si ya existe `BO_log.txt` en la carpeta del experimento, se archiva con fecha y hora antes de la BO, y producción verifica que el log tenga solo las 22 evaluaciones de esta corrida.

**Memoria:** los lags sobre 202103, 202104 y 202106 ocupan varios GB. Cerrá los otros kernels antes de correr.

## 1. Seteo del ambiente local

In [1]:
# instala solo lo que falta (la primera vez puede tardar)
paquetes <- c("data.table", "R.utils", "rlist", "bbotk", "mlr3", "mlr3mbo", "mlr3learners", "DiceKriging", "xgboost")
faltan <- paquetes[!sapply(paquetes, requireNamespace, quietly = TRUE)]
if (length(faltan) > 0) install.packages(faltan)

In [2]:
# limpio la memoria
rm(list = ls(all.names = TRUE)) # remove all objects
gc(full = TRUE, verbose = FALSE) # garbage collection

library(data.table)
library(R.utils)
library(rlist)
library(bbotk)
library(mlr3)
library(mlr3mbo)
library(mlr3learners)
# library(paradox)
library(DiceKriging)
library(xgboost)

RNGkind("L'Ecuyer-CMRG") # generador de la plantilla, fijo para todo el script
options(scipen = 999)

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2328100,124.4,4733517,252.8,2962444,158.3
Vcells,3796927,29.0,8388608,64.0,7530767,57.5


Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, warnings


Loading required package: paradox


Attaching package: ‘mlr3’


The following object is masked from ‘package:R.utils’:

    resample


Loading required package: mlr3

## 2. Definicion de Parametros

### 2.1 Leemos .yaml

In [3]:
# Leer el archivo de configuración.
config <- yaml::read_yaml("config.yaml",  eval.expr = TRUE)

In [4]:
PARAM <- list()
#directorios
# PARAM$data_dir <- config$directorios$datasets
# PARAM$exp_dir  <- config$directorios$exp

# LEEMOS DIRECCIONES RELATIVAS Y HACEMOS ABOLUTAS PARA EVITAR PROBLEMAS CON LOS "setwd()" en adelante.
PARAM$data_dir <- normalizePath(
  config$directorios$datasets, mustWork = TRUE
)
PARAM$exp_dir <- normalizePath(
  config$directorios$exp, mustWork = TRUE
)

PARAM$pred_dir <- normalizePath(
  config$directorios$predicciones, mustWork = TRUE
)

# training y future
PARAM$semillas <- config$training$semillas
PARAM$semilla_primigenia <- config$training$semilla_primigenia
PARAM$training_pct <- config$training$training_pct
PARAM$train <- config$training$train
PARAM$train_final <- config$training$train_final
PARAM$future <-  config$training$future
PARAM$cortes <- config$training$cortes 
PARAM$trainingstrategy$undersampling <- config$training$undersampling

# diapositiva: ksemillerio 1, repe 1, iteraciones 10
PARAM$hyperparametertuning$ksemillerio <- config$bayesian$ksemillerio    # con 1 y 1 hay una sola particion y una sola semilla en la busqueda
PARAM$hyperparametertuning$repe <- config$bayesian$repe
PARAM$hyperparametertuning$iteraciones <-config$bayesian$iteraciones      # iteraciones bayesianas
PARAM$hyperparametertuning$init <- config$bayesian$init                   # diseno inicial de mlr3mbo (4 x 3 hiperparametros), no cuenta como iteracion


In [5]:
# chequeamos q existan directorios
if (!dir.exists(PARAM$data_dir)) {
  stop("No existe el directorio de datasets: ", PARAM$data_dir)
}

if (!dir.exists(PARAM$exp_dir)) {
  stop("No existe el directorio de experimentos: ", PARAM$exp_dir)
}


In [6]:
# Nombre del experimento
PARAM$experimento <-config$bayesian$exp_name

In [7]:
# diapositiva: produccion con 10 semillas
PARAM$semillas_finales_n <- config$training$semillas_finales  

In [8]:
# Parametros XGBoost

# parametros fijos del XGBoost que se pisaran con la parte variable de la BO
PARAM$xgboost$param_fijos <- config$xgboost$fijos

Aqui se definen los hiperparámetros de XGBoost que participan de la Bayesian Optimization

In [9]:
# Aqui se cargan los bordes de los hiperparametros de la BO

# Construir el espacio usando todas las entradas de bayesian$hs.
PARAM$hypeparametertuning$hs <- do.call(
  paradox::ps,
  config$xgboost$hs
)
# PARAM$hypeparametertuning$hs2 

In [10]:
# particionar agrega una columna llamada fold a un dataset
#   que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(data, division, agrupa = "", campo = "fold", start = 1, seed = NA) {
  if (!is.na(seed)) set.seed(seed, "L'Ecuyer-CMRG")

  bloque <- unlist(mapply(
    function(x, y) {rep(y, x)}, division, seq(from = start, length.out = length(division))))

  data[, (campo) := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N], by = agrupa]
}

In [11]:
# logueo al archivo BO_log.txt
loguear <- function(preg, arch = NA, verbose = TRUE)
{
  t0 <- Sys.time()
  reg <- copy(preg)
  if ("feature_contri" %in% names(reg)) {
    reg$feature_contri <- reg$feature_contri[1]
  }
  archivo <- arch
  if (is.na(arch)) archivo <- paste0(folder, substitute(reg), ext)


  if (!file.exists(archivo))
  {
    # Escribo los titulos
    linea <- paste0("fecha\t",
                    paste(list.names(reg), collapse = "\t"), "\n")

    cat(linea, file = archivo)
  }

  # escribo el registro
  linea <- paste0(format(t0, "%Y%m%d.%H%M%S"), "\t",     # la fecha y hora
                  gsub(", ", "\t", toString(reg)), "\n")

  cat(linea, file = archivo, append = TRUE)  # grabo al archivo

  if (verbose) cat(linea)   # imprimo por pantalla
}

## 3. Generacion de la clase_ternaria (solo si no existe `competencia_02.csv`)

In [12]:
archivo_crudo    <- file.path(PARAM$data_dir, "competencia_02_crudo.csv")
archivo_ternaria <- file.path(PARAM$data_dir, "competencia_02.csv")

if (!file.exists(archivo_ternaria)) {
  if (!file.exists(archivo_crudo)) stop("No encuentro ni competencia_02.csv ni competencia_02_crudo.csv en ", PARAM$data_dir)

  dataset <- fread(archivo_crudo)

  # calculo el periodo0 consecutivo
  dsimple <- dataset[, list(
    "pos" = .I,
    numero_de_cliente,
    periodo0 = as.integer(foto_mes / 100) * 12 + foto_mes %% 100)]

  # ordeno
  setorder(dsimple, numero_de_cliente, periodo0)

  # calculo topes
  periodo_ultimo <- dsimple[, max(periodo0)]
  periodo_anteultimo <- periodo_ultimo - 1

  # calculo los leads de orden 1 y 2
  dsimple[, c("periodo1", "periodo2") :=
    shift(periodo0, n = 1:2, fill = NA, type = "lead"), numero_de_cliente]

  # assign most common class values = "CONTINUA"
  dsimple[periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA"]

  # calculo BAJA+1
  dsimple[periodo0 < periodo_ultimo &
    (is.na(periodo1) | periodo0 + 1 < periodo1),
    clase_ternaria := "BAJA+1"]

  # calculo BAJA+2
  dsimple[periodo0 < periodo_anteultimo & (periodo0 + 1 == periodo1)
    & (is.na(periodo2) | periodo0 + 2 < periodo2),
    clase_ternaria := "BAJA+2"]

  # pego el resultado en el dataset original y grabo
  setorder(dsimple, pos)
  dataset[, clase_ternaria := dsimple$clase_ternaria]
  fwrite(dataset, file = archivo_ternaria, sep = ",")
  rm(dataset, dsimple); gc()
}
cat("Uso:", archivo_ternaria, "\n")

Uso: /home/marco/code/DMEyF/datasets/competencia_02.csv 


In [13]:
# dataset <- fread(archivo_ternaria, stringsAsFactors = TRUE)
# sort(unique(dataset$foto_mes))

## 4. Preprocesamiento

In [14]:
# carpeta de trabajo
dir.create(file.path(PARAM$exp_dir, PARAM$experimento), showWarnings = FALSE, recursive = TRUE)
setwd(file.path(PARAM$exp_dir, PARAM$experimento))

# lectura del dataset (solo hasta el mes del future: no hace falta mas)
dataset <- fread(archivo_ternaria, stringsAsFactors = TRUE)
dataset <- dataset[foto_mes <= max(PARAM$future)]
stopifnot("clase_ternaria" %in% colnames(dataset))
setorder(dataset, numero_de_cliente, foto_mes)
cat("Meses:", paste(sort(unique(dataset$foto_mes)), collapse = ", "), "|", nrow(dataset), "filas x", ncol(dataset), "columnas\n")

Meses: 201901, 201902, 201903, 201904, 201905, 201906, 201907, 201908, 201909, 201910, 201911, 201912, 202001, 202002, 202003, 202004, 202005, 202006, 202007, 202008, 202009, 202010, 202011, 202012, 202101, 202102, 202103, 202104, 202105, 202106 | 4378500 filas x 155 columnas


### 4.1 Data quality
En cada `foto_mes`, la variable con **todos** sus valores exactamente en cero y **ningún** NA pasa a NA (como `ccajas_depositos` en 202105).

In [15]:
# campos_excluidos <- c("numero_de_cliente", "foto_mes", "clase_ternaria")
campos_excluidos <- config$training$campos_excluidos
campos_dq <- setdiff(colnames(dataset), campos_excluidos)
campos_dq <- campos_dq[sapply(dataset[, ..campos_dq], is.numeric)]

correcciones <- list()
for (mes in sort(unique(dataset$foto_mes))) {
  filas <- which(dataset$foto_mes == mes)
  for (campo in campos_dq) {
    x <- dataset[[campo]][filas]
    if (!anyNA(x) && all(x == 0)) {
      set(dataset, i = filas, j = campo, value = NA)
      correcciones[[length(correcciones) + 1]] <- data.table(foto_mes = mes, variable = campo, filas = length(filas))
    }
  }
}
correcciones <- rbindlist(correcciones)
cat("Variables pasadas a NA por mes:", nrow(correcciones), "\n")
if (nrow(correcciones) > 0) {
  print(correcciones)
  fwrite(correcciones, "data_quality_correcciones.csv")
}

Variables pasadas a NA por mes: 95 
    foto_mes                           variable  filas
       <int>                             <char>  <int>
 1:   201904  ctarjeta_visa_debitos_automaticos 126116
 2:   201904 mttarjeta_visa_debitos_automaticos 126116
 3:   201905                      mrentabilidad 126763
 4:   201905               mrentabilidad_annual 126763
 5:   201905                        mcomisiones 126763
 6:   201905                    mactivos_margen 126763
 7:   201905                    mpasivos_margen 126763
 8:   201905                  ccomisiones_otras 126763
 9:   201905                  mcomisiones_otras 126763
10:   201910                      mrentabilidad 135633
11:   201910               mrentabilidad_annual 135633
12:   201910                        mcomisiones 135633
13:   201910                    mactivos_margen 135633
14:   201910                    mpasivos_margen 135633
15:   201910        ccajeros_propios_descuentos 135633
16:   201910        mcajeros_

### 4.2 FE histórico: lags y delta lags de orden 1 y 2

In [16]:
campos_lag <- setdiff(colnames(dataset), campos_excluidos)
campos_lag <- campos_lag[sapply(dataset[, ..campos_lag], is.numeric)]
cat("Lags y deltas para", length(campos_lag), "variables\n")

t0 <- Sys.time()
dataset[, (paste0(campos_lag, "_lag1")) := shift(.SD, n = 1, fill = NA, type = "lag"), by = numero_de_cliente, .SDcols = campos_lag]
dataset[, (paste0(campos_lag, "_lag2")) := shift(.SD, n = 2, fill = NA, type = "lag"), by = numero_de_cliente, .SDcols = campos_lag]
for (campo in campos_lag) {
  dataset[, paste0(campo, "_delta1") := get(campo) - get(paste0(campo, "_lag1"))]
  dataset[, paste0(campo, "_delta2") := get(campo) - get(paste0(campo, "_lag2"))]
}
cat("Listo en", round(as.numeric(Sys.time() - t0, units = "secs"), 1), "seg |", ncol(dataset), "columnas\n")

Lags y deltas para 152 variables
Listo en 43.3 seg | 763 columnas


### 4.3 Meses de training y del future
De aca en adelante no se necesitan los otros meses. Como `train_final` es igual a `train`, `dataset_train` se reutiliza en la producción y se libera `dataset` (memoria).

In [17]:
# clase01: BAJA+1 y BAJA+2 son 1, CONTINUA es 0
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+2", "BAJA+1"), 1L, 0L)]

stopifnot(identical(sort(PARAM$train), sort(PARAM$train_final)))
dataset_train <- dataset[foto_mes %in% PARAM$train]
dfuture <- dataset[foto_mes %in% PARAM$future]   # datos del futuro, para aplicar el modelo
rm(dataset); gc(full = TRUE, verbose = FALSE)

dataset_train[, .N, list(foto_mes, clase_ternaria)]

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2416773,129.1,4733517,252.8,4733517,252.8
Vcells,2383457702,18184.4,5445259032,41544.1,4863948226,37109.0


foto_mes,clase_ternaria,N
<int>,<fct>,<int>
201901,CONTINUA,122572
201902,CONTINUA,123624
201903,CONTINUA,124148
201904,CONTINUA,124765
201905,CONTINUA,125529
201906,CONTINUA,127116
201907,CONTINUA,128602
201908,CONTINUA,130573
201909,CONTINUA,132256


In [18]:
particionar(dataset_train,
  division = c(PARAM$training_pct, 100L - PARAM$training_pct),
  agrupa = "clase_ternaria",
  seed = PARAM$semilla_primigenia # aqui se usa SU semilla
)

In [19]:
# defino los datos que forma parte del training
# aqui se hace el undersampling de los CONTINUA
# notar que para esto utilizo la SEGUNDA semilla

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset_train[, azar := runif(nrow(dataset_train))]
dataset_train[, training := 0L]

dataset_train[
  foto_mes %in% PARAM$train & fold == 1 &
    (azar <= PARAM$trainingstrategy$undersampling | clase_ternaria %in% c("BAJA+1", "BAJA+2")),
  training := 1L
]

In [20]:
# los campos que se van a utilizar

campos_buenos <- setdiff(
  colnames(dataset_train),
  c("clase_ternaria", "clase01", "fold", "azar", "training")
)
cat("Columnas para entrenar:", length(campos_buenos), "(la diapositiva dice 762)\n")

Columnas para entrenar: 762 (la diapositiva dice 762)


## 5. Bayesiana
### 5.1 Configuracion Bayesian Optimization

In [21]:
# Cada corrida de la BO arranca con un log limpio:
# si ya existe BO_log.txt, se archiva con fecha y hora en el nombre
giter <- 0

if (file.exists("BO_log.txt") && config$bayesian$correr_BO) {
  archivo_viejo <- paste0("BO_log_", format(Sys.time(), "%Y%m%d_%H%M%S"), ".txt")
  file.rename("BO_log.txt", archivo_viejo)
  cat("BO_log.txt anterior archivado como", archivo_viejo, "\n")
}

In [22]:
# En el argumento x llegan los parametros de la bayesiana

Estimar_xgboost <- function(xs) {

  flush.console()
  giter <<- giter + 1

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$xgboost$param_fijos, xs)
  xx <- copy(param_completo)

  # entreno XGBoost
  modelo <- xgb.train(
    data = dtrain,
    evals = list("val" = dvalidate),
    params = param_completo[setdiff(names(param_completo), "nrounds")],
    nrounds = as.integer(param_completo$nrounds),
    verbose = 0
  )


  # obtengo la metrica (AUC en validacion)
  xx$iter <- giter
  xx$metrica <- tail(attr(modelo, "evaluation_log")$val_auc, 1L)

  loguear(xx, "BO_log.txt")
  set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")  # le reordeno a mlr3MBO

  return(list(metrica = xx$metrica))
}

In [23]:
# dejo los datos de training en el formato que necesita XGBoost

dtrain <- xgb.DMatrix(
  data = data.matrix(dataset_train[training == 1L, campos_buenos, with = FALSE]),
  label = dataset_train[training == 1L, clase01]
)

nrow(dtrain)
ncol(dtrain)

[1] 305680

[1] 762

In [24]:
# datos de validacion (30% limpio, fold == 2) en el formato que necesita XGBoost
dvalidate <- xgb.DMatrix(
  data = data.matrix(dataset_train[fold == 2L, campos_buenos, with = FALSE]),
  label = dataset_train[fold == 2L, clase01]
)

nrow(dvalidate)
ncol(dvalidate)

[1] 1215180

[1] 762

In [25]:
# Aqui comienza la configuracion de la Bayesian Optimization

# en este archivo quedan la evolucion binaria de la BO
kbayesiana <- "bayesiana.RDATA"

funcion_optimizar <- Estimar_xgboost # la funcion que voy a maximizar

objective <- ObjectiveRFun$new(
  fun = funcion_optimizar,
  domain = PARAM$hypeparametertuning$hs,
  codomain = ps(metrica = p_dbl(tags = "maximize")),
  id = "BO_xgboost"
)


# diseno inicial (12) + iteraciones bayesianas (10) = 22 evaluaciones
instance <- OptimInstanceBatchSingleCrit$new(
  objective = objective,
  terminator = trm("evals", n_evals = PARAM$hyperparametertuning$init + PARAM$hyperparametertuning$iteraciones)
)


surrogate <- srlrn(lrn("regr.km",
  optim.method = "BFGS",
  nugget.estim = TRUE,
  control = list(trace = FALSE)))

acq_function <- acqf("ei")                 # expected improvement

acq_optimizer <- acqo(
  optimizer = opt("random_search", batch_size = 1000),
  terminator = trm("evals", n_evals = 1000)
)

optimizer <- opt("mbo",
  loop_function = bayesopt_ego,
  surrogate = surrogate,
  acq_function = acq_function,
  acq_optimizer = acq_optimizer
)

### 5.2 Corrida Bayesian Optimization

In [26]:
if (config$bayesian$correr_BO){
    bayesiana_salida <- optimizer$optimize(instance)
}

In [27]:
# control: deben ser init + iteraciones evaluaciones (22)
if (config$bayesian$correr_BO){
    cat("Evaluaciones de la BO:", nrow(instance$archive$data),
        "| esperadas:", PARAM$hyperparametertuning$init + PARAM$hyperparametertuning$iteraciones, "\n")
}

Luego de este punto, en la carpeta del experimento quedó el archivo BO_log.txt que posee todas las iteraciones de la Bayesian Optimization

## 6. Produccion
### 6.1 Final Training
Construyo el modelo final, que no hace ningun tipo de particion <training, validation, testing>

In [28]:
# cargo los mejores hiperparametros, que son los de mayor metrica
tb_bayesiana <- fread("BO_log.txt")
# control: el log tiene que tener solo las evaluaciones de esta corrida (init + iteraciones)
# stopifnot(nrow(tb_bayesiana) == PARAM$hyperparametertuning$init + PARAM$hyperparametertuning$iteraciones)
setorder(tb_bayesiana, -metrica)
PARAM$out$xgboost$mejores_hiperparametros <- as.list(tb_bayesiana[1])

param_final <- modifyList(PARAM$xgboost$param_fijos,
  PARAM$out$xgboost$mejores_hiperparametros)

# libero la memoria de la BO
rm(dtrain, dvalidate); gc(full = TRUE, verbose = FALSE)

print(tb_bayesiana[1, list(nrounds, max_leaves, max_depth, min_child_weight, learning_rate, auc_validacion = metrica)])

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2534592,135.4,4733517,252.8,4733517,252.8
Vcells,3557408838,27140.9,5445259032,41544.1,5074304719,38713.9


   nrounds max_leaves max_depth min_child_weight learning_rate auc_validacion
     <int>      <int>     <int>            <num>         <num>          <num>
1:    3193        830         0       0.02055577    0.02906926      0.9551553


In [29]:
PARAM$hyperparametertuning$init + PARAM$hyperparametertuning$iteraciones

[1] 26

In [30]:
nrow(tb_bayesiana)

[1] 16

### 6.2 Semillas del modelo final
La diapositiva pide 10 semillas. Son 10 numeros primos sorteados a partir de la semilla primigenia (siempre las mismas).

In [31]:
es_primo <- function(n) {
  if (n %% 2 == 0) return(FALSE)
  all(n %% seq(3, floor(sqrt(n)), by = 2) != 0)
}

set.seed(PARAM$semilla_primigenia)
semillas_finales <- integer(0)
while (length(semillas_finales) < PARAM$semillas_finales_n) {
  cand <- sample(100000:999999, 1)
  if (es_primo(cand) && !(cand %in% semillas_finales)) semillas_finales <- c(semillas_finales, cand)
}
semillas_finales

[1] 736381 445187 755311 339103 892559 925487 741493 796139 652357 822361

736381 445187 755311 339103 892559 925487 741493 796139 652357 822361


### 6.3 Entrenamiento final y prediccion de 202106
Por cada semilla: undersampling de los CONTINUA (0,1) sobre 202103+202104, modelo con los mejores hiperparametros y prediccion sobre 202106.

In [32]:
if (config$produccion$correr_prod){
  X_future <- data.matrix(dfuture[, campos_buenos, with = FALSE])
  prob_semillas <- matrix(NA_real_, nrow = nrow(X_future), ncol = length(semillas_finales),
    dimnames = list(as.character(dfuture$numero_de_cliente), paste0("S_", semillas_finales)))

  for (j in seq_along(semillas_finales)) {
    sem <- semillas_finales[j]
    t1 <- Sys.time()

    # undersampling de CONTINUA para el entrenamiento final (mismo 0.1 que en la BO)
    set.seed(sem, kind = "L'Ecuyer-CMRG")
    dataset_train[, azar := runif(.N)]
    dataset_train_sample <- dataset_train[
      azar <= PARAM$trainingstrategy$undersampling | clase_ternaria %in% c("BAJA+1", "BAJA+2")
    ]

    dtrain_final <- xgb.DMatrix(
      data = data.matrix(dataset_train_sample[, campos_buenos, with = FALSE]),
      label = dataset_train_sample[, clase01]
    )

    param_semilla <- param_final
    param_semilla$seed <- sem

    modelo_final <- xgb.train(
      data = dtrain_final,
      params = param_semilla[setdiff(names(param_semilla), "nrounds")],
      nrounds = as.integer(param_semilla$nrounds),
      verbose = 0
    )

    # aplico el modelo a los datos del futuro
    prob_semillas[, j] <- predict(modelo_final, X_future)
    rm(modelo_final, dtrain_final, dataset_train_sample); gc(verbose = FALSE)
    cat("Semilla", sem, "lista en", round(as.numeric(Sys.time() - t1, units = "mins"), 1), "min\n")
  }

  prob_final <- rowMeans(prob_semillas)
  saveRDS(prob_semillas, "prob_202106_por_semilla.rds")
}

In [33]:
# prob_semillas <- readRDS("prob_202106_por_semilla.rds")

## 7. Resultados a reportar
AUC, mejores hiperparámetros, ganancia máxima y cantidad de envíos.

In [34]:
if (config$produccion$correr_prod){
  auc_rank <- function(y, p) {
    r <- frank(p, ties.method = "average")
    n1 <- sum(y == 1); n0 <- sum(y == 0)
    (sum(r[y == 1]) - n1 * (n1 + 1) / 2) / (n1 * n0)
  }

  # ganancia de cada cliente del future (en el orden de dfuture)
  gan_cliente <- ifelse(dfuture$clase_ternaria == "BAJA+2", 1072500, -27500)

  # ganancia maxima de cada semilla, para ver la variabilidad
  gan_max_semilla <- sapply(seq_along(semillas_finales), function(j) max(cumsum(gan_cliente[order(-prob_semillas[, j])])))

  # calculo la ganancia en los datos del futuro para todos los cortes (ensamble de las semillas)
  tb_prediccion <- dfuture[, list(numero_de_cliente, foto_mes, clase_ternaria)]
  tb_prediccion[, prob := prob_final]
  tb_prediccion[, gan := ifelse(clase_ternaria == "BAJA+2", 1072500, -27500)]
  auc_test <- auc_rank(dfuture$clase01, prob_final)

  setorder(tb_prediccion, -prob)
  tb_prediccion[, ganancia_acumulada := cumsum(gan)]
  tb_prediccion[, pos := sequence(.N)]

  ganancia_max <- tb_prediccion[, max(ganancia_acumulada)]
  envios_max <- tb_prediccion[which.max(ganancia_acumulada), pos]

  cat("\n================ BASELINE: train 202103+202104, test 202106 ================\n")
  cat("Columnas para entrenar:        ", length(campos_buenos), "\n")
  cat("AUC en validacion (BO):        ", round(tb_bayesiana[1, metrica], 5), "\n")
  cat("AUC en 202106 (ensamble):      ", round(auc_test, 5), "(target BAJA+1 y BAJA+2)\n")
  cat("Mejores hiperparametros:        nrounds =", tb_bayesiana[1, nrounds],
      "| max_leaves =", tb_bayesiana[1, max_leaves],
      "| max_depth =", tb_bayesiana[1, max_depth],
      "| min_child_weight =", signif(tb_bayesiana[1, min_child_weight], 4),
      "| learning_rate =", signif(tb_bayesiana[1, learning_rate], 4), "\n")
  cat("GANANCIA MAXIMA (ensamble):    ", format(ganancia_max, big.mark = ".", scientific = FALSE), "\n")
  cat("CANTIDAD DE ENVIOS (maximo):   ", envios_max, "\n")
  cat("Ganancia maxima por semilla:    media", format(round(mean(gan_max_semilla)), big.mark = ".", scientific = FALSE),
      "| desvio", format(round(sd(gan_max_semilla)), big.mark = ".", scientific = FALSE), "\n")
}

In [35]:

# ganancia en los cortes de la plantilla y en el maximo
if (config$produccion$correr_prod){
  envios <- sort(unique(c(PARAM$cortes, envios_max)))
  tabla_gan <- data.table(
    envios = envios,
    ganancia = sapply(envios, function(e) tb_prediccion[1:e, sum(gan)])
  )
  tabla_gan[, ganancia_M := ganancia / 1e6]
  tabla_gan[, es_el_maximo := envios == envios_max]
  print(tabla_gan)

  fwrite(tabla_gan, "ganancia_por_envios.csv")
  fwrite(tb_prediccion[, list(pos, prob, gan, ganancia_acumulada)], "curva_ganancia.csv")
  fwrite(data.table(
    columnas = length(campos_buenos), auc_validacion_BO = tb_bayesiana[1, metrica], auc_202106 = auc_test,
    nrounds = tb_bayesiana[1, nrounds], max_leaves = tb_bayesiana[1, max_leaves],
    max_depth = tb_bayesiana[1, max_depth], learning_rate = tb_bayesiana[1, learning_rate],
    min_child_weight = tb_bayesiana[1, min_child_weight],
    ganancia_maxima = ganancia_max, envios_en_el_maximo = envios_max), "resumen.csv")
  cat("\nArchivos guardados en", getwd(), "\n")
}

In [36]:
if (config$produccion$correr_prod){
     plot(tb_prediccion$pos, tb_prediccion$ganancia_acumulada / 1e6, type = "l", lwd = 2, col = "#0F7B6C",
          xlab = "Cantidad de envios", ylab = "Ganancia (millones)", main = "Baseline: curva de ganancia en 202106")
     abline(v = envios_max, lty = 2, col = "#D9822B")
     legend("bottomright", legend = paste0("maximo: ", round(ganancia_max / 1e6, 1), " M con ", envios_max, " envios"), bty = "n")
}

In [37]:
if (config$produccion$correr_prod){
  # Tabla de referencia: ganancia en cortes FIJOS (ensamble y por semilla) en 202106
  cortes_ref <- sort(unique(c(PARAM$cortes, 10600, 12000)))

  # ganancia acumulada de cada semilla (filas = ranking, columnas = semillas)
  gan_acum_semilla <- sapply(seq_along(semillas_finales), function(j) {
    cumsum(gan_cliente[order(-prob_semillas[, j])])
  })

  tabla_ref <- data.table(
    envios            = cortes_ref,
    ensamble_M        = tb_prediccion$ganancia_acumulada[cortes_ref] / 1e6,
    semilla_media_M   = sapply(cortes_ref, function(e) mean(gan_acum_semilla[e, ])) / 1e6,
    semilla_sd_M      = sapply(cortes_ref, function(e) sd(gan_acum_semilla[e, ])) / 1e6,
    semilla_min_M     = sapply(cortes_ref, function(e) min(gan_acum_semilla[e, ])) / 1e6,
    semilla_max_M     = sapply(cortes_ref, function(e) max(gan_acum_semilla[e, ])) / 1e6
  )

  # rango de envios donde el ensamble queda a menos de 1 desvio por semilla del maximo
  sd_ref <- mean(tabla_ref$semilla_sd_M)
  max_ens <- max(tb_prediccion$ganancia_acumulada) / 1e6
  cat("Maximo del ensamble:", round(max_ens, 2), "M | desvio medio por semilla:", round(sd_ref, 2), "M\n")
  cat("Cortes a menos de 1 desvio del maximo:",
      paste(tabla_ref[ensamble_M >= max_ens - sd_ref, envios], collapse = ", "), "\n\n")

  print(tabla_ref[, lapply(.SD, round, 2), .SDcols = names(tabla_ref)])
  fwrite(tabla_ref, "tabla_referencia_cortes_fijos.csv")
}

## 8 Runner
    Creamos un Runner q se mantenga generando predicciones con distintas semillas y corte al alcanzar un limite establecido

    Tener en cuenta q para correr esta parte es necesario q previamente haya corrido la celda 6.1 (Final Training) donde se cargan los mejores HP para el experimento que estamos usando.


In [38]:
# Creo Semillas, misma logica previa para la seccion de produccion
if (config$predicciones$activar_runner){

    es_primo <- function(n) {
        if (n %% 2 == 0) return(FALSE)
        all(n %% seq(3, floor(sqrt(n)), by = 2) != 0)
    }
    set.seed(PARAM$semilla_primigenia)
    semillas_runner <- integer(0)
    while (length(semillas_runner) < config$predicciones$semillas_runner) {
        cand <- sample(100000:999999, 1)
        if (es_primo(cand) && !(cand %in% semillas_runner)) semillas_runner <- c(semillas_runner, cand)
    }
}

In [39]:
if (config$predicciones$activar_runner) {

  dir.create(PARAM$pred_dir, recursive = TRUE, showWarnings = FALSE)

  archivo <- file.path(
    PARAM$pred_dir,
    config$predicciones$predicciones_filename
  )

  ids <- as.character(dfuture$numero_de_cliente)
  columnas_semillas <- paste0("S_", semillas_runner)
  total <- length(semillas_runner)

  # Recupero las predicciones existentes y verifico clientes y orden.
  if (file.exists(archivo)) {

    predicciones <- data.table::fread(
      archivo,
      colClasses = c(ID = "character"),
      check.names = FALSE
    )

    if (!("ID" %in% names(predicciones)) ||
        !identical(predicciones$ID, ids)) {
      stop("El CSV existente no coincide con los clientes y el orden de dfuture.")
    }

    # Evito considerar completa una columna con predicciones faltantes.
    columnas_existentes <- intersect(columnas_semillas, names(predicciones))

    if (any(vapply(
      predicciones[, ..columnas_existentes],
      anyNA,
      logical(1)
    ))) {
      stop("El CSV contiene semillas con probabilidades faltantes.")
    }

  } else {
    predicciones <- data.table::data.table(ID = ids)
  }

  # Incorporar las predicciones terminadas de producción antes de entrenar.
  archivo_produccion <- "prob_202106_por_semilla.rds"
  if (file.exists(archivo_produccion)) {
    prob_produccion <- readRDS(archivo_produccion)
    if (!is.matrix(prob_produccion) ||
        !identical(rownames(prob_produccion), ids) ||
        is.null(colnames(prob_produccion)) ||
        anyDuplicated(colnames(prob_produccion))) {
      stop("El RDS de producción debe tener los mismos IDs y columnas S_<semilla> únicas.")
    }
    columnas_reusar <- setdiff(
      intersect(columnas_semillas, colnames(prob_produccion)),
      names(predicciones)
    )
    for (columna in columnas_reusar) {
      probabilidades <- prob_produccion[, columna]
      if (anyNA(probabilidades)) stop("Predicción incompleta en producción: ", columna)
      predicciones[, (columna) := probabilidades]
    }
    if (length(columnas_reusar) > 0L) {
      data.table::fwrite(predicciones, archivo)
      message(sprintf("Reutilizadas de producción | %d semillas", length(columnas_reusar)))
    }
    rm(prob_produccion)
  }

  # Solo entreno las semillas que todavía no tienen columna.
  pendientes <- which(!(columnas_semillas %in% names(predicciones)))

  # cat(
  #   "Predicciones guardadas |",
  #   total - length(pendientes), "de", total, "\n"
  # )
  message(sprintf(
    "Predicciones guardadas | %d de %d",
    total - length(pendientes), total
  ))

  if (length(pendientes) > 0L) {

    X_future <- data.matrix(dfuture[, campos_buenos, with = FALSE])

    for (j in pendientes) {

      sem <- semillas_runner[j]
      columna <- columnas_semillas[j]

      # Undersampling de CONTINUA, con la misma lógica anterior.
      set.seed(sem, kind = "L'Ecuyer-CMRG")
      dataset_train[, azar := runif(.N)]

      dataset_train_sample <- dataset_train[
        azar <= PARAM$trainingstrategy$undersampling |
          clase_ternaria %in% c("BAJA+1", "BAJA+2")
      ]

      dtrain_final <- xgb.DMatrix(
        data = data.matrix(
          dataset_train_sample[, campos_buenos, with = FALSE]
        ),
        label = dataset_train_sample[, clase01]
      )

      # Conservo los HP y cambio únicamente la semilla.
      param_semilla <- param_final
      param_semilla$seed <- sem

      modelo_final <- xgb.train(
        data = dtrain_final,
        params = param_semilla[setdiff(names(param_semilla), "nrounds")],
        nrounds = as.integer(param_semilla$nrounds),
        verbose = 0
      )

      # Agrego la columna y actualizo el CSV después de cada predicción.
      probabilidades <- predict(modelo_final, X_future)

      if (length(probabilidades) != length(ids) ||
          anyNA(probabilidades)) {
        stop("Predicción incompleta para la semilla ", sem)
      }

      predicciones[, (columna) := probabilidades]
      data.table::fwrite(predicciones, archivo)

      completas <- sum(columnas_semillas %in% names(predicciones))

      # cat(sprintf(
      #   "semilla %s completa | %d de %d | %.1f%%\n",
      #   sem, completas, total, 100 * completas / total
      # ))
      message(sprintf(
        "semilla %s completa | %d de %d | %.1f%%",
        sem, completas, total, 100 * completas / total
      ))
      rm(modelo_final, dtrain_final, dataset_train_sample, probabilidades)
      gc(verbose = FALSE)
    }
  }

}

Predicciones guardadas | 95 de 150

semilla 490559 completa | 96 de 150 | 64.0%

semilla 492511 completa | 97 de 150 | 64.7%

semilla 643031 completa | 98 de 150 | 65.3%

semilla 459337 completa | 99 de 150 | 66.0%

semilla 337793 completa | 100 de 150 | 66.7%

semilla 469757 completa | 101 de 150 | 67.3%

semilla 942113 completa | 102 de 150 | 68.0%

semilla 494251 completa | 103 de 150 | 68.7%

semilla 291143 completa | 104 de 150 | 69.3%

semilla 693607 completa | 105 de 150 | 70.0%

semilla 284701 completa | 106 de 150 | 70.7%

semilla 113209 completa | 107 de 150 | 71.3%

semilla 130043 completa | 108 de 150 | 72.0%

semilla 575699 completa | 109 de 150 | 72.7%

semilla 337511 completa | 110 de 150 | 73.3%

semilla 696359 completa | 111 de 150 | 74.0%

semilla 253433 completa | 112 de 150 | 74.7%

semilla 871883 completa | 113 de 150 | 75.3%

semilla 912991 completa | 114 de 150 | 76.0%

semilla 103591 completa | 115 de 150 | 76.7%

semilla 424771 completa | 116 de 150 | 77.3%

se

In [40]:
# Sanity check
prob_semillas <- readRDS("prob_202106_por_semilla.rds")
prob_semillas

,S_736381,S_445187,S_755311,S_339103,S_892559,S_925487,S_741493,S_796139,S_652357,S_822361
12159854,0.00000025913130,0.00000021437651,0.00000047168604,0.00000023529974,0.00000021074560,0.00000018451303,0.00000012943062,0.00000027947758,0.00000016251134,0.00000022687374
12159858,0.00001776092904,0.00005440670429,0.00001003498255,0.00004161038669,0.00003625664613,0.00001855716255,0.00008961222920,0.00007710918726,0.00003896226917,0.00003119349276
12160484,0.00000858391195,0.00000682883865,0.00000341954865,0.00000147184517,0.00000208469896,0.00000276136734,0.00000402334354,0.00000394120434,0.00000365097549,0.00000486092085
12160591,0.00000021775870,0.00000009794415,0.00000037970108,0.00000015625956,0.00000032444399,0.00000014149668,0.00000022465095,0.00000023522233,0.00000024869510,0.00000010470062
12160747,0.00015431540669,0.00002751634565,0.00005006158972,0.00003299141827,0.00013099594798,0.00011513707432,0.00004051119686,0.00007830067625,0.00022700070986,0.00011389105202
12160769,0.00000602800310,0.00002331079668,0.00000976191586,0.00000443892350,0.00000550491768,0.00000957128850,0.00000649950471,0.00000783136329,0.00000461165746,0.00000962646936
12160968,0.00061230600113,0.00008537164831,0.00035606688471,0.00008713554416,0.00015100176097,0.00023076555226,0.00072727439692,0.00016544740356,0.00059904210502,0.00042500934796
12161135,0.00000133137132,0.00000224424912,0.00000085391480,0.00000066985962,0.00000221568644,0.00000072629678,0.00000080922172,0.00000216587159,0.00000127501323,0.00000106785353
12161416,0.00325111416169,0.00404459750280,0.00382564100437,0.00619794009253,0.01011930033565,0.01468977518380,0.00231498735957,0.00582536682487,0.00340713164769,0.00707647530362
12161961,0.00000018944769,0.00000016045038,0.00000048758017,0.00000036588315,0.00000044474137,0.00000046366168,0.00000086950877,0.00000023337780,0.00000052770088,0.00000070637401


In [41]:
pred_csv <- data.table::fread(file.path(
  PARAM$pred_dir,
  config$predicciones$predicciones_filename
))

prob_semillas <- readRDS("prob_202106_por_semilla.rds")

# Segunda columna del CSV: la primera es ID.
all.equal(
  pred_csv[[2L]],
  prob_semillas[, 1L],
  tolerance = 0,
  check.attributes = FALSE
)

[1] "Mean relative difference: 0.000000000000000463819"

In [42]:
dim(pred_csv)
dim(prob_semillas)

[1] 164114    151

[1] 164114     10